# BDDK C=0,01 Kapasite-Düşürülmüş Tekrar — Ders Kitabı Notebooku

Model 13, Model 12'nin zayıf işaretini tek bir daha güçlü düzenlileştirme noktasında terminal olarak sınar. Bu in-sample/permutasyon heuristiği OOF performans veya temiz-vintaj kanıtı değildir.

## Okuma hedefleri

- C=0,01 manipülasyonunun ezber null95'ini gerçekten düşürüp düşürmediğini görmek.
- 10-feature kontrol ile 14-feature BDDK kolunda gözlenen ve null95 artışlarını ayırmak.
- Pozitif delta marjı mutlak kol2 marjıyla aynı anda yorumlamak.
- Yalnız yeni C=0,01 çiftinin terminal kararı verdiğini doğrulamak.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
repo = Path.cwd()
if not (repo / 'data').exists(): repo = repo.parent
ozet = json.loads((repo / 'data/processed/model/model_13_bddk_c001_ozet.json').read_text(encoding='utf-8'))
print(f"{ozet['model']} | {ozet['karar']['hukum']} / {ozet['karar']['tarama_kesinligi']} | test={ozet['test']}")

Model 13 BDDK C=0,01 kapasite-dusuk tekrar | KAPASITE_DUSUK_ISARET_YOK / HEURISTIK | test=2025-07..2026-06 ACILMADI_KILITLI


## Ön-kapılar

Özgün dört Model 12 konfigürasyonu kontrol harness'ini ve test-kolu yeniden üretimini geçmek zorundadır. C=0,01 yeni baseline'dır; manipülasyon kapısı `null95(C=0,01, kontrol) < 0,4450343895977828` koşuludur.

In [2]:
h = pd.DataFrame(ozet['harness_ozgun_dort']).T
t = pd.DataFrame(ozet['model12_test_tekrar_denetimi']).T
display(h[['en_buyuk_mutlak_fark', 'gecti']])
display(t[['gecti']])
display(pd.DataFrame([ozet['karar']['manipulasyon_kapisi']]))
assert h['gecti'].all() and t['gecti'].all() and ozet['karar']['manipulasyon_kapisi']['gecti']

,en_buyuk_mutlak_fark,gecti
lojistik_l2_c01,0.0,True
lojistik_l2_c1,0.0,True
random_forest_sigin,0.0,True
hist_gradient_sigin,0.0,True


,gecti
lojistik_l2_c01,True
lojistik_l2_c1,True
random_forest_sigin,True
hist_gradient_sigin,True


,olculen_null95,strict_esik,gecti
0,0.421986,0.445034,True


## Kapasite ve iki kol

Kontrol kolu 10, BDDK'lı kol 14 feature kullanır; nominal feature sayısı artışı %40'tır. Null95 artışı ek kapasitenin karıştırılmış etiketleri ezberleme artışını, gözlenen artış gerçek etiket uyumundaki artışı gösterir. Karar için bunların farkı olan delta marj tek başına yeterli değildir; mutlak kol2 marjı da aynı anda okunur.

In [3]:
k = pd.DataFrame(ozet['karsilastirma']).T
sutunlar = ['kol1_gozlenen','kol2_gozlenen','gozlenen_degisim','kol1_null95','kol2_null95','null95_degisim','kol1_marj','kol2_marj','delta_marj','yeni_baseline']
display(k[sutunlar].style.format(precision=4))
c = k.loc['lojistik_l2_c001']
display(Markdown(f"C=0,01: gözlenen artış **{c.gozlenen_degisim:+.4f}**, null95 artışı **{c.null95_degisim:+.4f}**, delta marj **{c.delta_marj:+.4f}**; ancak mutlak kol2 marjı **{c.kol2_marj:+.4f}**."))

,kol1_gozlenen,kol2_gozlenen,gozlenen_degisim,kol1_null95,kol2_null95,null95_degisim,kol1_marj,kol2_marj,delta_marj,yeni_baseline
lojistik_l2_c01,0.2148,0.3794,0.1646,0.4450,0.5006,0.0555,-0.2303,-0.1211,0.1091,False
lojistik_l2_c1,0.1690,0.4936,0.3246,0.4684,0.5528,0.0844,-0.2994,-0.0592,0.2402,False
random_forest_sigin,0.9169,0.9690,0.0522,0.9155,0.9537,0.0382,0.0013,0.0153,0.0140,False
hist_gradient_sigin,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,False
lojistik_l2_c001,0.2137,0.2713,0.0577,0.4220,0.4528,0.0308,-0.2083,-0.1815,0.0268,True


C=0,01: gözlenen artış **+0.0577**, null95 artışı **+0.0308**, delta marj **+0.0268**; ancak mutlak kol2 marjı **-0.1815**.

## Terminal karar

Kapı sırası: manipülasyon etkili mi → kol2 marjı +0,15'i geçiyor mu → delta marj +0,15'i geçiyor mu. Özgün C=1 zayıf işareti bu karara yeniden girmez; daha fazla C taraması yasaktır.

In [4]:
karar = ozet['karar']
display(Markdown(f"**Hüküm:** `{karar['hukum']}` / `{karar['tarama_kesinligi']}`  \n**Sonraki dal:** `{karar['otomatik_sonraki_dal']}`  \n**Yeniden açma önceliği:** `{karar['yeniden_acma_onceligi']}`  \n**Daha fazla C taraması:** `{karar['daha_fazla_c_taramasi']}`"))

**Hüküm:** `KAPASITE_DUSUK_ISARET_YOK` / `HEURISTIK`  
**Sonraki dal:** `YENI_MASA_BASI_TARAMASI_NORMAL_YENIDEN_ACMA`  
**Yeniden açma önceliği:** `NORMAL`  
**Daha fazla C taraması:** `False`

## Yorum sınırı

`KAPASITE_DUSUK_ISARET_YOK`, BDDK'nın ekonomik olarak sinyalsiz olduğunu veya temiz vintajın başarısız olacağını kanıtlamaz. Yalnız C=0,01 tekrarının ön-kayıtlı mutlak/delta kapılarını geçmediğini gösterir. BDDK normal yeniden-açma önceliğiyle `ONCELIK_DUSURULDU`; kilitli test açılmadı.